In [4]:
import os
import numpy as np
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D # GlobalAveragePooling2D만 사용하거나, 512 Dense까지 포함
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# --- 1. 설정 (기존과 동일) ---
IMG_WIDTH, IMG_HEIGHT = 300, 300
BATCH_SIZE = 32
N_CLASSES = 3 # 비빔밥, 햄버거, 스시
CLASS_LABELS = ['ramen', 'udon', 'pasta']

# 훈련 및 테스트 데이터 경로 (이전에 나눈 train/test 폴더 사용)
TRAIN_DATA_DIR = 'E:/★★★★★AI★★★★★/음식 ai data/selectStart음식DATA/Computer Vision Lab/train' # YOUR_TRAIN_DIR_PATH
TEST_DATA_DIR = 'E:/★★★★★AI★★★★★/음식 ai data/selectStart음식DATA/Computer Vision Lab/test'   # YOUR_TEST_DIR_PATH

# --- 2. 특징 추출기 모델 로드 ---
# EfficientNetB3를 특징 추출기로 사용
base_model = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(IMG_WIDTH, IMG_HEIGHT, 3))

# GlobalAveragePooling2D 레이어까지만 사용하거나,
# 만약 Dense(512) 레이어가 더 나은 특징을 준다고 생각하면 그 레이어까지 포함할 수 있습니다.
# 여기서는 GlobalAveragePooling2D의 출력인 1280차원 벡터를 특징으로 사용한다고 가정합니다.
feature_extractor_model = Model(inputs=base_model.input, outputs=GlobalAveragePooling2D()(base_model.output))

# 베이스 모델의 가중치는 동결되어 있어야 합니다.
# (EfficientNetB3를 include_top=False로 불러왔고, 우리가 추가하는 레이어가 없으므로 따로 trainble=False 설정 불필요)

# --- 3. 데이터 제너레이터 설정 (레이블만 필요) ---
# 특징 추출을 위해 ImageDataGenerator를 사용하되, shuffle=False로 고정하고 batch_size를 크게 해도 무방합니다.
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DATA_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE, # 특징 추출 시 BATCH_SIZE는 메모리에 맞게 조절
    class_mode='categorical', # 레이블을 얻기 위함
    shuffle=False # 특징 추출 시 순서 고정
)

test_generator = test_datagen.flow_from_directory(
    TEST_DATA_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False # 특징 추출 시 순서 고정
)

print(f"Found {train_generator.samples} training images.")
print(f"Found {test_generator.samples} test images.")

# --- 4. 훈련 데이터셋에서 특징 추출 ---
print("\nExtracting features from training images...")
X_train_features = feature_extractor_model.predict(train_generator,
                                                   steps=train_generator.samples // BATCH_SIZE + (1 if train_generator.samples % BATCH_SIZE else 0),
                                                   verbose=1)
y_train = train_generator.classes # 정수 레이블 (0, 1, 2...)

print(f"Extracted {X_train_features.shape[0]} training features with shape {X_train_features.shape[1]}")

# --- 5. 테스트 데이터셋에서 특징 추출 ---
print("\nExtracting features from test images...")
X_test_features = feature_extractor_model.predict(test_generator,
                                                  steps=test_generator.samples // BATCH_SIZE + (1 if test_generator.samples % BATCH_SIZE else 0),
                                                  verbose=1)
y_test = test_generator.classes # 정수 레이블 (0, 1, 2...)

print(f"Extracted {X_test_features.shape[0]} test features with shape {X_test_features.shape[1]}")


# --- 6. 랜덤 포레스트 모델 학습 ---
print("\nTraining Random Forest Classifier...")
# n_estimators: 트리의 개수, random_state: 재현성을 위한 시드
rf_model = RandomForestClassifier(n_estimators=100, random_state=47, n_jobs=-1)
rf_model.fit(X_train_features, y_train)

print("\nRandom Forest Training Complete!")

# --- 7. 랜덤 포레스트 모델 평가 ---
print("\nEvaluating Random Forest Classifier...")
y_pred = rf_model.predict(X_test_features)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nRandom Forest Test Accuracy: {accuracy*100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_LABELS))

Found 1009 images belonging to 3 classes.
Found 440 images belonging to 3 classes.
Found 1009 training images.
Found 440 test images.

Extracting features from training images...
32/32 [==============================] - 20s 576ms/step
Extracted 1009 training features with shape 1536

Extracting features from test images...
14/14 [==============================] - 8s 579ms/step
Extracted 440 test features with shape 1536

Training Random Forest Classifier...

Random Forest Training Complete!

Evaluating Random Forest Classifier...

Random Forest Test Accuracy: 55.23%

Classification Report:
              precision    recall  f1-score   support

       ramen       0.53      0.64      0.58       135
        udon       0.00      0.00      0.00       102
       pasta       0.57      0.77      0.65       203

    accuracy                           0.55       440
   macro avg       0.36      0.47      0.41       440
weighted avg       0.42      0.55      0.48       440



C:\Users\user\anaconda3\envs\ml-dl-nlp\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\user\anaconda3\envs\ml-dl-nlp\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\user\anaconda3\envs\ml-dl-nlp\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
